# 🌊 Partitions - Grid 1 (Single Point)

This notebook processes wave spectra partitioning for a specific point.

In [ ]:
import os
import numpy as np
import xarray as xr
import wavespectra  # registers .spec accessor

# ============================================================================
# Simplified processing: 4 grids, 1 point per grid, all variables, all years
# Output: one NetCDF per variable in `outputs/`, with 4 points
# ============================================================================

# Region / output configuration
REGION_NAME = "NorthCarolina" 
OUTPUT_DIR = "outputs/partitions_SuperPoint/"
BULK_OUTPUT_DIR = "outputs/partitions_SuperPoint/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BULK_OUTPUT_DIR, exist_ok=True)

# Input spectrum: one file per grid (from 01_BinWaves reconstructed spectra)
BINWAVES_DIR = os.path.abspath("../../01_BinWaves")
INPUT_SPECTRA_FILES = {
    "grid1": os.path.join(BINWAVES_DIR, "grid1/inputs/grid1_41013_spec_WHACS_buoy_correted_15D.nc"),
    "grid2": os.path.join(BINWAVES_DIR, "grid2/inputs/grid2_superPoint_15D.nc"),
    "grid3": os.path.join(BINWAVES_DIR, "grid3/inputs/grid3_superPoint_15D.nc"),
    "grid4": os.path.join(BINWAVES_DIR, "grid4/inputs/grid4_44014_spec_WHACS_buoy_correted_15D.nc"),
}

UWND_FILE = "inputs/uwnd.nc"
VWND_FILE = "inputs/vwnd.nc"
# wind data from WHACS
GEBCO_FILE = os.path.join(BINWAVES_DIR, "inputs/gebco_bathymetry.nc")

# Target points: 1 per grid (lat, lon)
TARGET_POINTS = {
    "grid1": {"label": "grid1", "lat": 33.441,  "lon": -77.766},
    "grid2": {"label": "grid2", "lat": 33.8928, "lon": -76.9845},
    "grid3": {"label": "grid3", "lat": 35.10,   "lon": -75.36},
    "grid4": {"label": "grid4", "lat": 36.603,  "lon": -74.837},
}

# ----------------------------------------------------------------------------
# Load forcing datasets (wind and bathymetry)
# ----------------------------------------------------------------------------

def open_netcdf_with_fallback(path: str, fallback_paths=None):
    """Open NetCDF trying explicit engines and optional fallback paths."""
    candidates = [path]
    if fallback_paths:
        candidates.extend(fallback_paths)

    last_err = None
    for p in candidates:
        if not os.path.exists(p):
            continue
        for eng in ["netcdf4", "scipy", None]:
            try:
                if eng is None:
                    return xr.open_dataset(p)
                return xr.open_dataset(p, engine=eng)
            except Exception as exc:
                last_err = exc
                continue

    if last_err is not None:
        raise RuntimeError(f"Could not open any candidate NetCDF file for {path}. Last error: {last_err}")
    raise FileNotFoundError(f"None of the candidate files exist for {path}. Checked: {candidates}")


print("Loading wind components (WHACS)...")
uwnd = open_netcdf_with_fallback(
    UWND_FILE,
    fallback_paths=["inputs/WHACS/uwnd_NorthCarolina_closest4.nc"],
)
vwnd = open_netcdf_with_fallback(
    VWND_FILE,
    fallback_paths=["inputs/WHACS/vwnd_NorthCarolina_closest4.nc"],
)

uwnd_var = [v for v in uwnd.data_vars if "uwnd" in v.lower() or v.lower() == "u" or v.lower().startswith("u_")][0]
vwnd_var = [v for v in vwnd.data_vars if "vwnd" in v.lower() or v.lower() == "v" or v.lower().startswith("v_")][0]

print(f"  U wind variable: {uwnd_var}")
print(f"  V wind variable: {vwnd_var}")

print("\nLoading GEBCO bathymetry...")
gebco = open_netcdf_with_fallback(GEBCO_FILE)
print("  GEBCO loaded.")

# ----------------------------------------------------------------------------
# Helper: wind seapoint coordinates (1D lon/lat arrays for WHACS seapoints)
# ----------------------------------------------------------------------------

print("\nPreparing wind seapoint coordinate arrays (for nearest-neighbour search)...")
uwnd_filtered = uwnd.sel(time=slice("1980-01-01", None))  # 1979 coords may be problematic

lat_coord = None
lon_coord = None
for c in uwnd_filtered.coords:
    lc = c.lower()
    if "lat" in lc:
        lat_coord = c
    if "lon" in lc:
        lon_coord = c

# If not in coords, check data_vars
if lat_coord is None:
    for v in uwnd_filtered.data_vars:
        if "lat" in v.lower():
            lat_coord = v
            break
if lon_coord is None:
    for v in uwnd_filtered.data_vars:
        if "lon" in v.lower():
            lon_coord = v
            break

if lat_coord is None or lon_coord is None:
    raise ValueError(f"Could not find lat/lon in wind dataset. Coords: {list(uwnd_filtered.coords)}, vars: {list(uwnd_filtered.data_vars)}")

lat_da = uwnd_filtered[lat_coord]
lon_da = uwnd_filtered[lon_coord]

# Build 1D arrays (one coordinate per seapoint) robustly for both:
# - closest4 files: (seapoint, time)
# - other files:    (time, seapoint) or already (seapoint,)
if "seapoint" in lat_da.dims:
    sel_index = {d: 0 for d in lat_da.dims if d != "seapoint"}
    uwnd_lat = lat_da.isel(**sel_index).values
    uwnd_lon = lon_da.isel(**sel_index).values
else:
    # Fallback: flatten and use as-is (legacy behavior)
    uwnd_lat = lat_da.values.ravel()
    uwnd_lon = lon_da.values.ravel()

print(f"  Wind seapoints: {uwnd_lon.size} points")


def find_closest_seapoint(lon_deg: float, lat_deg: float) -> int:
    """Return index of closest WHACS seapoint to (lon, lat)."""
    # Convert to 0–360 longitude if needed
    lon360 = lon_deg + 360 if lon_deg < 0 else lon_deg

    lon_diff = np.abs(uwnd_lon - lon360)
    lon_diff = np.minimum(lon_diff, 360 - lon_diff)  # handle wrap-around
    lat_diff = np.abs(uwnd_lat - lat_deg)

    dist2 = lon_diff ** 2 + lat_diff ** 2
    return int(np.argmin(dist2))


def distance_km_to_seapoint(lon_deg: float, lat_deg: float, seapoint_idx: int) -> float:
    """Approximate great-circle distance [km] between target and WHACS seapoint."""
    # Target in degrees
    lat1 = np.radians(lat_deg)
    lon1 = np.radians(lon_deg)

    # Seapoint lon is in 0–360; convert back to -180..180 for distance
    seap_lon_360 = float(uwnd_lon[seapoint_idx])
    seap_lon = seap_lon_360 if seap_lon_360 <= 180.0 else seap_lon_360 - 360.0
    lat2 = np.radians(float(uwnd_lat[seapoint_idx]))
    lon2 = np.radians(seap_lon)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    R_earth_km = 6371.0
    return float(R_earth_km * c)


def distance_km_between_points(lat1_deg: float, lon1_deg: float, lat2_deg: float, lon2_deg: float) -> float:
    """Great-circle distance [km] between two (lat, lon) points in degrees."""
    lat1 = np.radians(lat1_deg)
    lon1 = np.radians(lon1_deg)
    lat2 = np.radians(lat2_deg)
    lon2 = np.radians(lon2_deg)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    R_earth_km = 6371.0
    return float(R_earth_km * c)


# ----------------------------------------------------------------------------
# Helper: single-depth extraction from GEBCO for a (lon, lat) pair
# ----------------------------------------------------------------------------

def get_depth_from_gebco(lon_deg: float, lat_deg: float) -> float:
    """Return (positive) water depth [m] from GEBCO nearest to (lon, lat)."""
    # Find depth/elevation variable
    depth_var = None
    for var in gebco.data_vars:
        vl = var.lower()
        if "elevation" in vl or "depth" in vl or "bathymetry" in vl or vl == "z":
            depth_var = var
            break
    if depth_var is None:
        depth_var = list(gebco.data_vars.keys())[0]

    # Find coordinate names
    gebco_lon = None
    gebco_lat = None
    for coord in gebco.coords:
        cl = coord.lower()
        if "lon" in cl or cl == "x":
            gebco_lon = coord
        if "lat" in cl or cl == "y":
            gebco_lat = coord

    if gebco_lon is None or gebco_lat is None:
        raise ValueError("Could not find lon/lat coordinates in GEBCO dataset")

    depth = gebco[depth_var]
    if depth.min() < 0:
        depth = np.abs(depth)

    depth_at_point = depth.sel(
        {gebco_lon: lon_deg, gebco_lat: lat_deg},
        method="nearest",
    )
    return float(depth_at_point.values)


# ----------------------------------------------------------------------------
# Helper: wind speed/direction for one seapoint, aligned to spectra time
# ----------------------------------------------------------------------------

def get_wind_for_times(time_index: xr.DataArray, seapoint_idx: int):
    """Return wspd, wdir as DataArrays with dimension 'time' aligned to spectra time."""
    t_min = str(time_index.min().values)
    t_max = str(time_index.max().values)

    uwnd_sel = uwnd.sel(time=slice(t_min, t_max))
    vwnd_sel = vwnd.sel(time=slice(t_min, t_max))

    u_site = uwnd_sel[uwnd_var].isel(seapoint=seapoint_idx)
    v_site = vwnd_sel[vwnd_var].isel(seapoint=seapoint_idx)

    wspd = np.sqrt(u_site ** 2 + v_site ** 2)
    # Wind direction: oceanographic convention, 0–360°
    wdir = (np.arctan2(u_site, v_site) * 180.0 / np.pi + 180.0) % 360.0

    wspd = wspd.reindex(time=time_index, method="nearest")
    wdir = wdir.reindex(time=time_index, method="nearest")

    return wspd, wdir


# ----------------------------------------------------------------------------
# Helper: PTM4 partitioning for a single site (no Dask, simple & robust)
# ----------------------------------------------------------------------------

def partition_single_site(spectra_site: xr.Dataset, wspd: xr.DataArray, wdir: xr.DataArray) -> xr.DataArray:
    """Run PTM4 partitioning for a single spatial point, returning a DataArray.

    spectra_site: Dataset with variables 'efth' (time, freq, dir) and 'dpt' (time)
    wspd, wdir:   wind speed/direction DataArrays with dimension 'time'
    """
    dpt = spectra_site["dpt"]

    # Ensure only dimension coordinates (drop scalar coords that confuse wavespectra)
    def _clean(var: xr.DataArray) -> xr.DataArray:
        drop = [c for c in var.coords if c not in var.dims]
        return var.drop_vars(drop) if drop else var

    wspd_c = _clean(wspd)
    wdir_c = _clean(wdir)
    dpt_c = _clean(dpt)

    # Align times exactly
    target_time = spectra_site.efth["time"]
    wspd_c = wspd_c.reindex(time=target_time, method="nearest")
    wdir_c = wdir_c.reindex(time=target_time, method="nearest")
    dpt_c = dpt_c.reindex(time=target_time, method="nearest")

    # PTM1: one wind sea + `swells` swell systems. swells=1 -> 2 partitions total.
    dspart = spectra_site.spec.partition.ptm4(
        wspd_c,
        wdir_c,
        dpt_c,
        # swells=1,
        # smooth=False,
    )
    # wavespectra here returns a DataArray (efth with an extra 'part' dim).
    # Ensure it is fully in-memory (no dask chunks).
    if hasattr(dspart, "chunks") and dspart.chunks is not None:
        dspart = dspart.load()
    return dspart


# ----------------------------------------------------------------------------
# Processing loop over 4 grids / 4 points
# ----------------------------------------------------------------------------

PART_VARS = ["hs", "tp", "dp", "spr"]  # significant height, peak period, peak dir, spreading
BULK_VARS = ["hs", "tp", "dp"]           # full-spectrum bulk variables

# Metadata for partitioned outputs (used for per-year and per-grid files)
GRID_PART_OUTPUTS = {
    "hs":  {"varname": "phs",  "long_name": "partition_significant_wave_height", "units": "m"},
    "tp":  {"varname": "ptp",  "long_name": "partition_peak_period", "units": "s"},
    "dp":  {"varname": "pdp",  "long_name": "partition_peak_direction", "units": "degrees"},
    "spr": {"varname": "spr",  "long_name": "partition_directional_spreading", "units": "degrees"},
}

GRID_BULK_OUTPUTS = {
    "hs": {"varname": "hs", "long_name": "bulk_significant_wave_height", "units": "m"},
    "tp": {"varname": "tp", "long_name": "bulk_peak_period", "units": "s"},
    "dp": {"varname": "dp", "long_name": "bulk_peak_direction", "units": "degrees"},
}

# Accumulators per grid/point
per_grid_results = {}

grid_order = sorted(TARGET_POINTS.keys())


def _get_efth_dataset(ds: xr.Dataset) -> xr.Dataset:
    """Ensure dataset has 'efth' variable (spectra). Handles common naming conventions."""
    if "efth" in ds.data_vars:
        return ds
    # Try common alternatives
    for name in ["__xarray_dataarray_variable__", "spec", "energy", "e"]:
        if name in ds.data_vars:
            return ds.rename({name: "efth"})
    raise ValueError(f"Could not find spectra variable. Available: {list(ds.data_vars)}")


def _get_single_site_spectra(ds: xr.Dataset, lat: float, lon: float) -> xr.Dataset:
    """Extract spectra for target point: use closest site if multi-site, else use as-is."""
    ds = _get_efth_dataset(ds)
    # Find spatial dimension (site, point, station, etc.)
    site_dims = [d for d in ds.efth.dims if d not in ("time", "freq", "dir", "direction")]
    if not site_dims:
        # Single point: no site dimension
        spectra_site = ds.drop_vars([v for v in ds.data_vars if v != "efth"], errors="ignore")
    else:
        site_dim = site_dims[0]
        n_sites = ds.sizes[site_dim]
        if n_sites == 1:
            spectra_site = ds.isel({site_dim: 0}).drop_vars(
                [v for v in ds.data_vars if v != "efth"], errors="ignore")
        else:
            # Find closest site by coord_x/coord_y or lat/lon
            lon_c = next((c for c in ds.coords if "lon" in c.lower() or "x" in c.lower()), None)
            lat_c = next((c for c in ds.coords if "lat" in c.lower() or "y" in c.lower()), None)
            if lon_c is None or lat_c is None:
                raise ValueError("Multi-site dataset needs coord_x/coord_y or lat/lon")
            lons_s = ds[lon_c].values
            lats_s = ds[lat_c].values
            if lons_s.ndim > 1:
                lons_s, lats_s = lons_s[0], lats_s[0]
            dist2 = (lons_s - lon) ** 2 + (lats_s - lat) ** 2
            site_idx = int(np.argmin(dist2))
            spectra_site = ds.isel({site_dim: site_idx}).drop_vars(
                [v for v in ds.data_vars if v != "efth"], errors="ignore")
    return spectra_site


def process_single_grid(grid_name: str):
    """Process one grid (one target point) using its dedicated input spectrum file."""
    cfg = TARGET_POINTS[grid_name]
    lat = cfg["lat"]
    lon = cfg["lon"]
    label = cfg["label"]

    print("\n" + "=" * 72)
    print(f"Processing {grid_name} ({label}) at lat={lat:.4f}, lon={lon:.4f}")
    print("=" * 72)

    fpath = INPUT_SPECTRA_FILES.get(grid_name)
    if not fpath or not os.path.exists(fpath):
        raise FileNotFoundError(f"Input spectrum not found for {grid_name}: {fpath}")

    print(f"  Input: {fpath}")

    # Precompute nearest wind seapoint and bathymetric depth for this location
    seapoint_idx = find_closest_seapoint(lon, lat)
    dist_km = distance_km_to_seapoint(lon, lat, seapoint_idx)
    depth_val = get_depth_from_gebco(lon, lat)
    print(f"  Closest wind seapoint index: {seapoint_idx}")
    print(f"  Distance to closest seapoint: {dist_km:.2f} km")
    print(f"  Bathymetric depth: {depth_val:.2f} m")

    # Check for existing per-grid checkpoints (partition + bulk)
    checkpoint_files_part = {
        src_name: os.path.join(OUTPUT_DIR, f"{meta['varname']}_{grid_name}.nc")
        for src_name, meta in GRID_PART_OUTPUTS.items()
    }
    checkpoint_files_bulk = {
        src_name: os.path.join(BULK_OUTPUT_DIR, f"{meta['varname']}_{grid_name}.nc")
        for src_name, meta in GRID_BULK_OUTPUTS.items()
    }
    if all(os.path.exists(p) for p in checkpoint_files_part.values()) and all(os.path.exists(p) for p in checkpoint_files_bulk.values()):
        print("  Checkpoint files found (partition + bulk), loading instead of recomputing...")
        part_merged = {
            name: xr.open_dataset(checkpoint_files_part[name])[GRID_PART_OUTPUTS[name]["varname"]]
            for name in PART_VARS
        }
        bulk_merged = {
            name: xr.open_dataset(checkpoint_files_bulk[name])[GRID_BULK_OUTPUTS[name]["varname"]]
            for name in BULK_VARS
        }
        return {"label": label, "lat": lat, "lon": lon, "part": part_merged, "bulk": bulk_merged}

    print("  Loading spectra file...")
    ds = xr.open_dataset(fpath)
    spectra_site = _get_single_site_spectra(ds, lat, lon)

    # Attach depth as 1D time series (constant in time)
    depth_da = xr.DataArray(
        np.full(spectra_site.time.size, depth_val, dtype="float32"),
        dims=["time"],
        coords={"time": spectra_site.time},
        name="dpt",
    )
    spectra_site = spectra_site.assign(dpt=depth_da)

    # Full-spectrum bulk parameters (same site/time used for partitioning)
    print("  Computing full-spectrum bulk parameters...")
    spec_full = spectra_site.spec
    bulk_merged = {
        "hs": spec_full.hs(),
        "tp": spec_full.tp(),
        "dp": spec_full.dpm(),
    }

    wspd, wdir = get_wind_for_times(spectra_site.time, seapoint_idx)

    print("  Running PTM4 partitioning...")
    dspart_site = partition_single_site(spectra_site, wspd, wdir)

    n_parts = int(min(2, dspart_site.sizes.get("part", dspart_site.part.size)))
    part_indices = list(range(n_parts))

    hs_parts, tp_parts, dp_parts, spr_parts = [], [], [], []
    for p in part_indices:
        part_ds = xr.Dataset({"efth": dspart_site.isel(part=p)})
        spec_part = part_ds.spec
        hs_parts.append(spec_part.hs())
        tp_parts.append(spec_part.tp())
        dp_parts.append(spec_part.dpm())
        spr_parts.append(spec_part.dspr())

    phs_file = xr.concat(hs_parts, dim="part").assign_coords(part=part_indices)
    ptp_file = xr.concat(tp_parts, dim="part").assign_coords(part=part_indices)
    pdp_file = xr.concat(dp_parts, dim="part").assign_coords(part=part_indices)
    spr_file = xr.concat(spr_parts, dim="part").assign_coords(part=part_indices)

    part_merged = {"hs": phs_file, "tp": ptp_file, "dp": pdp_file, "spr": spr_file}

    ds.close()
    del spectra_site, dspart_site, hs_parts, tp_parts, dp_parts, spr_parts, spec_full, wspd, wdir
    import gc
    gc.collect()

    # Save per-grid checkpoint files
    print("  Saving per-grid checkpoint files (time, part)...")
    for src_name, meta in GRID_PART_OUTPUTS.items():
        varname = meta["varname"]
        da = part_merged[src_name]  # dims: time, part
        ds_out = xr.Dataset({varname: da})
        ds_out.attrs.update({
            "region": REGION_NAME,
            "grid": grid_name,
            "label": label,
            "description": f"PTM4-partitioned {meta['long_name']} at single point for {grid_name}",
        })
        fname = os.path.join(OUTPUT_DIR, f"{varname}_{grid_name}.nc")
        ds_out.to_netcdf(fname, encoding={varname: {"zlib": True, "complevel": 4, "shuffle": True}})

    print("  Saving per-grid bulk checkpoint files (time)...")
    for src_name, meta in GRID_BULK_OUTPUTS.items():
        varname = meta["varname"]
        da = bulk_merged[src_name]  # dims: time
        ds_out = xr.Dataset({varname: da})
        ds_out.attrs.update({
            "region": REGION_NAME,
            "grid": grid_name,
            "label": label,
            "description": f"Full-spectrum {meta['long_name']} at single point for {grid_name}",
        })
        fname = os.path.join(BULK_OUTPUT_DIR, f"{varname}_{grid_name}.nc")
        ds_out.to_netcdf(fname, encoding={varname: {"zlib": True, "complevel": 4, "shuffle": True}})

    # Return results for this grid
    return {
        "label": label,
        "lat": lat,
        "lon": lon,
        "part": part_merged,
        "bulk": bulk_merged,
    }


# Process the four grids sequentially (more stable on limited-memory kernels)
print("\nProcessing 4 grids sequentially...")

for grid_name in grid_order:
    per_grid_results[grid_name] = process_single_grid(grid_name)


# ----------------------------------------------------------------------------
# Combine 4 points into single arrays and write one NetCDF per variable
# ----------------------------------------------------------------------------

print("\nCombining all 4 grid points and writing NetCDF outputs...")

# Use time from the first grid as reference (from partitioned hs, first part)
ref_grid = grid_order[0]
ref_time = per_grid_results[ref_grid]["part"]["hs"].isel(part=0).time

point_labels = [per_grid_results[g]["label"] for g in grid_order]
point_lats = [per_grid_results[g]["lat"] for g in grid_order]
point_lons = [per_grid_results[g]["lon"] for g in grid_order]

# Helper to stack per-grid partition data along new 'point' dimension

def stack_over_points(name: str) -> xr.DataArray:
    """Stack partitioned per-grid results for variable `name` over a new 'point' dimension.

    Result dims: (time, point, part)
    """
    arrays = []
    for g in grid_order:
        da = per_grid_results[g]["part"][name].copy()
        # Drop non-dimension coords that differ between grids
        drop = [c for c in da.coords if c not in da.dims]
        if drop:
            da = da.drop_vars(drop)
        # Align time to reference (just in case)
        da = da.reindex(time=ref_time, method="nearest")
        da = da.expand_dims(point=[per_grid_results[g]["label"]])
        arrays.append(da)
    return xr.concat(arrays, dim="point", compat="no_conflicts")


def stack_bulk_over_points(name: str) -> xr.DataArray:
    """Stack bulk per-grid results for variable `name` over a new 'point' dimension.

    Result dims: (time, point)
    """
    arrays = []
    for g in grid_order:
        da = per_grid_results[g]["bulk"][name].copy()
        drop = [c for c in da.coords if c not in da.dims]
        if drop:
            da = da.drop_vars(drop)
        da = da.reindex(time=ref_time, method="nearest")
        da = da.expand_dims(point=[per_grid_results[g]["label"]])
        arrays.append(da)
    return xr.concat(arrays, dim="point", compat="no_conflicts")


# --- Partitioned variables ---
# We write one file per physical quantity, with dims (time, point, part).
part_outputs = {
    "hs":  {"varname": "phs",  "long_name": "partition_significant_wave_height", "units": "m"},
    "tp":  {"varname": "ptp",  "long_name": "partition_peak_period", "units": "s"},
    "dp":  {"varname": "pdp",  "long_name": "partition_peak_direction", "units": "degrees"},
    "spr": {"varname": "spr",  "long_name": "partition_directional_spreading", "units": "degrees"},
}

for src_name, meta in part_outputs.items():
    da = stack_over_points(src_name)  # dims: time, point, part
    varname = meta["varname"]
    ds_out = xr.Dataset({varname: da})
    ds_out = ds_out.assign_coords(
        lat=("point", point_lats),
        lon=("point", point_lons),
    )
    ds_out[varname].attrs.update({"long_name": meta["long_name"], "units": meta["units"]})
    ds_out.attrs.update({
        "region": REGION_NAME,
        "description": f"PTM4-partitioned {meta['long_name']} at 4 selected points (one per grid)",
    })

    fname = os.path.join(OUTPUT_DIR, f"{varname}_{REGION_NAME}.nc")
    print(f"  Writing {fname} ...")
    ds_out.to_netcdf(fname, encoding={varname: {"zlib": True, "complevel": 4, "shuffle": True}})

bulk_outputs = {
    "hs": {"varname": "hs", "long_name": "bulk_significant_wave_height", "units": "m"},
    "tp": {"varname": "tp", "long_name": "bulk_peak_period", "units": "s"},
    "dp": {"varname": "dp", "long_name": "bulk_peak_direction", "units": "degrees"},
}

for src_name, meta in bulk_outputs.items():
    da = stack_bulk_over_points(src_name)  # dims: time, point
    varname = meta["varname"]
    ds_out = xr.Dataset({varname: da})
    ds_out = ds_out.assign_coords(
        lat=("point", point_lats),
        lon=("point", point_lons),
    )
    ds_out[varname].attrs.update({"long_name": meta["long_name"], "units": meta["units"]})
    ds_out.attrs.update({
        "region": REGION_NAME,
        "description": f"Full-spectrum {meta['long_name']} at 4 selected points (one per grid)",
    })

    fname = os.path.join(BULK_OUTPUT_DIR, f"{varname}_{REGION_NAME}.nc")
    print(f"  Writing {fname} ...")
    ds_out.to_netcdf(fname, encoding={varname: {"zlib": True, "complevel": 4, "shuffle": True}})

print("\nPartition outputs written to:", OUTPUT_DIR)
print("Bulk outputs written to:", BULK_OUTPUT_DIR)

Loading wind components (WHACS)...
  U wind variable: uwnd
  V wind variable: vwnd

Loading GEBCO bathymetry...
  GEBCO loaded.

Preparing wind seapoint coordinate arrays (for nearest-neighbour search)...
  Wind seapoints: 2572 points

Processing 4 grids sequentially...

Processing grid1 (grid1) at lat=33.4410, lon=-77.7660
  Input: /lustre/geocean/WORK/users/montanoj/personal/ShoreShop2026/grid1/inputs/grid1_41013_spec_WHACS_buoy_correted_15D.nc
  Closest wind seapoint index: 569
  Distance to closest seapoint: 1.53 km
  Bathymetric depth: 27.00 m
  Loading spectra file...
  Computing full-spectrum bulk parameters...
  Running PTM4 partitioning...
  Saving per-grid checkpoint files (time, part)...
  Saving per-grid bulk checkpoint files (time)...

Processing grid2 (grid2) at lat=33.8928, lon=-76.9845
  Input: /lustre/geocean/WORK/users/montanoj/personal/ShoreShop2026/grid2/inputs/grid2_superPoint_15D.nc
  Closest wind seapoint index: 842
  Distance to closest seapoint: 2.44 km
  Bathy